In [1]:
# Cài đặt PySpark
%pip install pyspark

In [2]:
# Import các thư viện cần thiết
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
import os

# Khởi tạo Spark Context
conf = SparkConf().setAppName("MovieRatingsAnalysis").setMaster("local[*]")
sc = SparkContext.getOrCreate(conf=conf)
spark = SparkSession.builder.appName("MovieRatingsAnalysis").getOrCreate()

print("Spark Context đã được khởi tạo thành công!")

Spark Context đã được khởi tạo thành công!


In [3]:
# Đọc dữ liệu từ các file
import os

# Check if running in Google Colab
if 'COLAB_GPU' in os.environ or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    data_path = "/content/"
else:
    data_path = "data/" # For local environment

# Đọc file movies.txt
movies_rdd = sc.textFile(data_path + "movies.txt")
print(f"Số lượng phim: {movies_rdd.count()}")

# Đọc file ratings_1.txt và ratings_2.txt
ratings_1_rdd = sc.textFile(data_path + "ratings_1.txt")
ratings_2_rdd = sc.textFile(data_path + "ratings_2.txt")

print(f"Số lượng rating từ file 1: {ratings_1_rdd.count()}")
print(f"Số lượng rating từ file 2: {ratings_2_rdd.count()}")

# Hiển thị một số dòng dữ liệu mẫu
print("\nDữ liệu movies.txt (5 dòng đầu):")
for line in movies_rdd.take(5):
    print(line)

print("\nDữ liệu ratings_1.txt (5 dòng đầu):")
for line in ratings_1_rdd.take(5):
    print(line)

Số lượng phim: 50
Số lượng rating từ file 1: 84
Số lượng rating từ file 2: 100

Dữ liệu movies.txt (5 dòng đầu):
1001,The Godfather (1972),Crime|Drama
1002,The Shawshank Redemption (1994),Drama
1003,Schindler's List (1993),Biography|Drama|History
1004,Raging Bull (1980),Biography|Drama|Sport
1005,Casablanca (1942),Drama|Romance|War

Dữ liệu ratings_1.txt (5 dòng đầu):
7,1020,4.5,1577836800
23,1015,3.5,1577923200
45,1030,4.0,1578009600
12,1047,3.0,1578096000
38,1012,4.5,1578182400


In [4]:
# Xử lý dữ liệu movies
# Parse movies.txt: MovieID, Title, Genres
def parse_movie_with_genres(line):
    parts = line.split(',', 2)  # Tách thành 3 phần: ID, Title, Genres
    movie_id = int(parts[0])
    title = parts[1]
    genres = parts[2] if len(parts) > 2 else ""
    # Tách các thể loại bằng dấu "|"
    genre_list = [genre.strip() for genre in genres.split('|') if genre.strip()]
    return (movie_id, title, genre_list)

movies_parsed = movies_rdd.map(parse_movie_with_genres)
print("Movies parsed with genres (5 records):")
for movie in movies_parsed.take(5):
    print(f"MovieID: {movie[0]}, Title: {movie[1]}, Genres: {movie[2]}")

# Tạo dictionary để tra cứu thông tin phim theo ID (bao gồm genres)
movies_dict = movies_parsed.map(lambda x: (x[0], (x[1], x[2]))).collectAsMap()
print(f"\nTổng số phim trong dictionary: {len(movies_dict)}")

# Tạo RDD chứa (movie_id, genre) cho mỗi thể loại của mỗi phim
movie_genres = movies_parsed.flatMap(lambda x: [(x[0], genre) for genre in x[2]])
print(f"\nTổng số cặp (movie_id, genre): {movie_genres.count()}")

print("\nVí dụ movie_genres (10 records):")
for mg in movie_genres.take(10):
    print(f"MovieID: {mg[0]}, Genre: {mg[1]}")

Movies parsed with genres (5 records):
MovieID: 1001, Title: The Godfather (1972), Genres: ['Crime', 'Drama']
MovieID: 1002, Title: The Shawshank Redemption (1994), Genres: ['Drama']
MovieID: 1003, Title: Schindler's List (1993), Genres: ['Biography', 'Drama', 'History']
MovieID: 1004, Title: Raging Bull (1980), Genres: ['Biography', 'Drama', 'Sport']
MovieID: 1005, Title: Casablanca (1942), Genres: ['Drama', 'Romance', 'War']

Tổng số phim trong dictionary: 50

Tổng số cặp (movie_id, genre): 122

Ví dụ movie_genres (10 records):
MovieID: 1001, Genre: Crime
MovieID: 1001, Genre: Drama
MovieID: 1002, Genre: Drama
MovieID: 1003, Genre: Biography
MovieID: 1003, Genre: Drama
MovieID: 1003, Genre: History
MovieID: 1004, Genre: Biography
MovieID: 1004, Genre: Drama
MovieID: 1004, Genre: Sport
MovieID: 1005, Genre: Drama


In [5]:
# Xử lý dữ liệu ratings
# Parse ratings: UserID, MovieID, Rating, Timestamp
def parse_rating(line):
    parts = line.split(',')
    user_id = int(parts[0])
    movie_id = int(parts[1])
    rating = float(parts[2])
    timestamp = int(parts[3])
    return (movie_id, rating)

# Parse cả 2 file ratings
ratings_1_parsed = ratings_1_rdd.map(parse_rating)
ratings_2_parsed = ratings_2_rdd.map(parse_rating)

print("Ratings 1 parsed (5 records):")
for rating in ratings_1_parsed.take(5):
    print(f"MovieID: {rating[0]}, Rating: {rating[1]}")

print("\nRatings 2 parsed (5 records):")
for rating in ratings_2_parsed.take(5):
    print(f"MovieID: {rating[0]}, Rating: {rating[1]}")

# Gộp 2 RDD ratings lại
all_ratings = ratings_1_parsed.union(ratings_2_parsed)
print(f"\nTổng số ratings từ cả 2 file: {all_ratings.count()}")

Ratings 1 parsed (5 records):
MovieID: 1020, Rating: 4.5
MovieID: 1015, Rating: 3.5
MovieID: 1030, Rating: 4.0
MovieID: 1047, Rating: 3.0
MovieID: 1012, Rating: 4.5

Ratings 2 parsed (5 records):
MovieID: 1012, Rating: 3.5
MovieID: 1039, Rating: 4.0
MovieID: 1043, Rating: 4.5
MovieID: 1020, Rating: 3.0
MovieID: 1050, Rating: 4.0

Tổng số ratings từ cả 2 file: 184


In [6]:
# Tính điểm trung bình và tổng số lượt đánh giá cho từng thể loại

# Join ratings với movie_genres để có (genre, rating)
# all_ratings có format (movie_id, rating)
# movie_genres có format (movie_id, genre)

# Join để có (movie_id, (rating, genre))
ratings_with_genres = all_ratings.join(movie_genres)
print("Ratings with genres (5 records):")
for record in ratings_with_genres.take(5):
    print(f"MovieID: {record[0]}, Rating: {record[1][0]}, Genre: {record[1][1]}")

# Chuyển đổi thành (genre, rating)
genre_ratings = ratings_with_genres.map(lambda x: (x[1][1], x[1][0]))

print(f"\nTổng số genre-rating pairs: {genre_ratings.count()}")
print("\nGenre ratings (10 records):")
for gr in genre_ratings.take(10):
    print(f"Genre: {gr[0]}, Rating: {gr[1]}")

# Tính tổng rating và số lượng rating cho mỗi thể loại
# (genre, rating) -> (genre, (rating, 1))
genre_ratings_with_count = genre_ratings.map(lambda x: (x[0], (x[1], 1)))

# Reduce theo key để tính tổng rating và tổng số lượng rating cho mỗi thể loại
# (genre, (sum_ratings, total_count))
genre_stats = genre_ratings_with_count.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

print(f"\nTổng số thể loại có rating: {genre_stats.count()}")
print("Genre stats (5 records):")
for stat in genre_stats.take(5):
    print(f"Genre: {stat[0]}, Sum: {stat[1][0]}, Count: {stat[1][1]}")

Ratings with genres (5 records):
MovieID: 1020, Rating: 4.5, Genre: Family
MovieID: 1020, Rating: 4.5, Genre: Sci-Fi
MovieID: 1020, Rating: 3.5, Genre: Family
MovieID: 1020, Rating: 3.5, Genre: Sci-Fi
MovieID: 1020, Rating: 3.5, Genre: Family

Tổng số genre-rating pairs: 471

Genre ratings (10 records):
Genre: Family, Rating: 4.5
Genre: Sci-Fi, Rating: 4.5
Genre: Family, Rating: 3.5
Genre: Sci-Fi, Rating: 3.5
Genre: Family, Rating: 3.5
Genre: Sci-Fi, Rating: 3.5
Genre: Family, Rating: 3.5
Genre: Sci-Fi, Rating: 3.5
Genre: Family, Rating: 3.5
Genre: Sci-Fi, Rating: 3.5

Tổng số thể loại có rating: 12
Genre stats (5 records):
Genre: Sci-Fi, Sum: 201.5, Count: 54
Genre: Action, Sum: 200.5, Count: 54
Genre: Drama, Sum: 481.0, Count: 128
Genre: Family, Sum: 66.0, Count: 18
Genre: Biography, Sum: 89.0, Count: 25


In [7]:
# Tính điểm trung bình cho từng thể loại và hiển thị kết quả
def calculate_genre_average(record):
    genre, (sum_ratings, count) = record
    average_rating = sum_ratings / count
    return (genre, (average_rating, count))

genre_results = genre_stats.map(calculate_genre_average)

all_genres = genre_results.collect()
for genre, (avg_rating, count) in all_genres:
    print(f"{genre} - AverageRating: {avg_rating:.2f} (TotalRatings: {count})")

Sci-Fi - AverageRating: 3.73 (TotalRatings: 54)
Action - AverageRating: 3.71 (TotalRatings: 54)
Drama - AverageRating: 3.76 (TotalRatings: 128)
Family - AverageRating: 3.67 (TotalRatings: 18)
Biography - AverageRating: 3.56 (TotalRatings: 25)
Horror - AverageRating: 4.00 (TotalRatings: 2)
Fantasy - AverageRating: 3.86 (TotalRatings: 29)
Thriller - AverageRating: 3.70 (TotalRatings: 27)
Mystery - AverageRating: 4.00 (TotalRatings: 2)
Adventure - AverageRating: 3.63 (TotalRatings: 83)
Film-Noir - AverageRating: 4.36 (TotalRatings: 7)
Crime - AverageRating: 3.81 (TotalRatings: 42)


In [8]:
# Dọn dẹp tài nguyên
sc.stop()
spark.stop()
print("Đã dừng Spark Context và Spark Session.")

Đã dừng Spark Context và Spark Session.
